In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# load data
df = pd.read_csv("traffic_analysis.csv")

# create binary outcome
df["citation_binary"] = df["citation_issued"].fillna(False).astype(int)

# keep only needed columns
model_df = df[[
    "citation_binary",
    "subject_age",
    # "subject_race" Not including because too many missing
    "subject_sex",
    "economic_diversity_index",
    "median_household_income",
    "poverty_rate"
]].copy()

# recode sex
model_df["subject_sex"] = model_df["subject_sex"].str.strip().str.lower()
model_df["subject_sex"] = model_df["subject_sex"].map({"male": 0, "female": 1})

# split predictors and response
X = model_df.drop(columns="citation_binary")
y = model_df["citation_binary"]


# impute missing values with median
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# fit logistic regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# predicted probabilities
y_pred_prob = log_model.predict_proba(X_test)[:, 1]


/var/folders/kt/_bzbz3vd5qbcfjnt5vssmqxh0000gn/T/ipykernel_16546/1356110672.py:8: DtypeWarning: Columns (12,13,17,18,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("traffic_analysis.csv")
/var/folders/kt/_bzbz3vd5qbcfjnt5vssmqxh0000gn/T/ipykernel_16546/1356110672.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["citation_binary"] = df["citation_issued"].fillna(False).astype(int)
/Users/janak/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/janak/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matm

In [2]:
print(y.value_counts(normalize=True))

citation_binary
0    0.681749
1    0.318251
Name: proportion, dtype: float64


In [3]:
# ROC AUC
roc_auc = roc_auc_score(y_test, y_pred_prob)
print("ROC AUC:", roc_auc)

ROC AUC: 0.6763540396391186


In [4]:
print(X.columns)

Index(['subject_age', 'subject_sex', 'economic_diversity_index',
       'median_household_income', 'poverty_rate'],
      dtype='object')
